# Gravitational attention study: Colab runner

Runs every phase of the study on a Colab GPU (an A100 was used for the paper) and writes all outputs to Google Drive, so a phase can be resumed after a disconnect. Each run is skipped if its `results.json` already exists.

Before starting, put the four e-SNLI CSV files (`esnli_train_1.csv`, `esnli_train_2.csv`, `esnli_dev.csv`, `esnli_test.csv`, from https://github.com/OanaMariaCamburu/e-SNLI) in a Drive folder `e-SNLI` inside `BASE`.

In [ ]:
# 1. Mount Drive, get the code, install requirements
from google.colab import drive
import os, subprocess, sys
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/GAM_Research'   # outputs go here; change if you like
os.makedirs(BASE, exist_ok=True)
if not os.path.exists('/content/gravitational-attention'):
    subprocess.run(['git', 'clone', 'https://github.com/ctheradyubele/gravitational-attention.git', '/content/gravitational-attention'], check=True)
os.chdir('/content/gravitational-attention')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Code ready')

In [ ]:
# 2. Point the protocol at Drive and link the e-SNLI data
import yaml
if not os.path.exists('data/esnli'):
    os.makedirs('data', exist_ok=True)
    os.symlink(f'{BASE}/e-SNLI', 'data/esnli')
p = yaml.safe_load(open('configs/protocol.yaml'))
p['runs_dir'] = BASE
yaml.safe_dump(p, open('configs/protocol_final.yaml', 'w'), sort_keys=False)
os.environ['GAM_PROTOCOL'] = os.path.abspath('configs/protocol_final.yaml')
os.environ['USE_TF'] = '0'
print('runs ->', BASE)

In [ ]:
# 3. Unit tests (all 34 must pass)
!python -m pytest -q tests

In [ ]:
# 4. Helper: run every command of a phase, logging to Drive
import torch
print(torch.cuda.get_device_name(0))
def run_phase(*phases):
    name = ''.join(phases)
    with open(f'cmds_{name}.txt', 'w') as f:
        for ph in phases:
            f.write(subprocess.run([sys.executable, 'scripts/run_matrix.py', ph], capture_output=True, text=True, check=True).stdout)
    open(f'run_{name}.sh', 'w').write(f'''#!/bin/bash
while read -r c; do
  echo "== $(date +%H:%M) $c" | cut -c1-160
  eval "$c" >> "{BASE}/{name}_log.txt" 2>&1 || echo "FAILED: $c" >> "{BASE}/failed.txt"
done < cmds_{name}.txt
''')
    subprocess.run(['bash', f'run_{name}.sh'])

In [ ]:
# 5. Phase A: continued MLM pretraining (7 variants)
run_phase('A')

In [ ]:
# 6. Phase F: time and memory benchmark
run_phase('F')

In [ ]:
# 7. Phase B1: learning-rate selection (144 runs)
run_phase('B1')

In [ ]:
# 8. Phase B2: GLUE final runs and SST-2 ablations (250 runs)
run_phase('B2')

In [ ]:
# 9. Phases C and D: e-SNLI explanations and MNLI training-set size
run_phase('C', 'D')

In [ ]:
# 10. Analysis and figures
!python analysis/aggregate.py --runs {BASE} --out {BASE}/results 2>&1 | tail -n 40
!python analysis/mass_analysis.py --runs {BASE} --esnli_dir data/esnli --out {BASE}/results 2>&1 | tail -n 20
!cp {BASE}/results/* results/ && python figures/make_figures.py && ls figures